In [1]:
print("""
@File         : pd.DataFrame.explode.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-04 14:23:07
@Email        : cuixuanstephen@gmail.com
@Description  : pd.DataFrame.explode
""")


@File         : pd.DataFrame.explode.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-04 14:23:07
@Email        : cuixuanstephen@gmail.com
@Description  : pd.DataFrame.explode



In [2]:
import pandas as pd

如果每条数据都以标量的形式完美地融入二维 pd.DataFrame 中，世界就会变得如此简单。可惜，生活并没有那么简单。特别是在处理 JSON 等半结构化数据源时，pd.DataFrame 中的个别项目包含列表和元组等非标量序列的情况并不少见。

![Using pd.DataFrame.explode to extract list elements to individual rows](../../IMAGES/FIG7-7.png)

In [3]:
df = pd.DataFrame(
    [
        {
            "employee_id": 1,
            "first_name": "John",
            "last_name": "Smith",
            "direct_reports": [2, 3]
        },
        {
            "employee_id": 2,
            "first_name": "Jane",
            "last_name": "Doe",
            "direct_reports": []
        },
        {
            "employee_id": 3,
            "first_name": "Joe",
            "last_name": "Schmoe",
            "direct_reports": []
        }
    ]
)
df = df.convert_dtypes(dtype_backend="numpy_nullable")
df

,employee_id,first_name,last_name,direct_reports
0,1,John,Smith,"[2, 3]"
1,2,Jane,Doe,[]
2,3,Joe,Schmoe,[]


使用 `pd.DataFrame.explode`，可以将这些 `direct_reports` 解压到 pd.DataFrame 的单独行中：

In [5]:
df.explode('direct_reports')

,employee_id,first_name,last_name,direct_reports
0,1,John,Smith,2
0,1,John,Smith,3
1,2,Jane,Doe,NaN
2,3,Joe,Schmoe,NaN


In [8]:
pd.merge(df.explode('direct_reports'), df.drop(columns=['direct_reports']),
         how='left', left_on=['direct_reports'], right_on=['employee_id'],
         suffixes=['', '_direct_reports'])

,employee_id,first_name,last_name,direct_reports,employee_id_direct_reports,first_name_direct_reports,last_name_direct_reports
0,1,John,Smith,2,2,Jane,Doe
1,1,John,Smith,3,3,Joe,Schmoe
2,2,Jane,Doe,NaN,<NA>,<NA>,<NA>
3,3,Joe,Schmoe,NaN,<NA>,<NA>,<NA>


In [9]:
import pyarrow as pa

dtype = pd.ArrowDtype(pa.struct([
    ("int_col", pa.int64()),
    ("str_col", pa.string()),
    ("float_col", pa.float64()),
]))
ser = pd.Series([
    {"int_col": 42, "str_col": "Hello, ", "float_col": 3.14159},
    {"int_col": 555, "str_col": "world!", "float_col": 3.14159},
], dtype=dtype)
ser

0    {'int_col': 42, 'str_col': 'Hello, ', 'float_c...
1    {'int_col': 555, 'str_col': 'world!', 'float_c...
dtype: struct<int_col: int64, str_col: string, float_col: double>[pyarrow]

与生成新数据行的 `pd.DataFrame.explode` 不同，`pd.Series.struct.explode` 从其结构成员中生成新数据列：

In [10]:
ser.struct.explode()

,int_col,str_col,float_col
0,42,"Hello,",3.14159
1,555,world!,3.14159


This could be particularly useful if you are dealing with a semi-structured data source like JSON. If you are able to fit nested data from such a source into the typed struct that PyArrow has to offer, `pd.Series.struct.explode` can save you a significant amount of trouble when trying to unnest that data.